# 第20章：FlashAttention v1 — 分块 + 在线 Softmax

## 本章概览

在上一章中，我们看到标准 Attention 需要 O(N^2) 的内存来存储注意力矩阵 S。
**FlashAttention v1** 的核心洞察是：

> **永远不要在 HBM 中物化完整的 N x N 注意力矩阵。**

它通过两个关键技术实现了这一点：
1. **分块计算（Tiling）**：将 Q, K, V 分成小块，在 SRAM 中完成计算
2. **在线 Softmax（Online Softmax）**：在不知道完整行的情况下，逐块更新 softmax

本章我们将：
1. 理解在线 Softmax 的数学原理
2. 理解 FlashAttention v1 的算法（外层遍历 K,V 块，内层遍历 Q 块）
3. 用 Triton 实现 FlashAttention v1 前向传播
4. 支持因果掩码
5. 验证正确性并进行性能测试

## 1. 在线 Softmax 原理

### 1.1 标准 Softmax 的问题

标准数值稳定 Softmax 需要 **三遍遍历**：

```
第 1 遍: m = max(x)              # 找到最大值
第 2 遍: l = sum(exp(x - m))     # 求和
第 3 遍: y = exp(x - m) / l      # 归一化
```

这意味着需要完整的行数据才能开始计算。
但在 FlashAttention 中，我们一次只能看到一个 K 块产生的部分分数。

### 1.2 在线 Softmax（Online Softmax）

在线 Softmax 允许我们**逐块更新** softmax 结果，不需要一次看到完整的行。

假设我们已经处理了前 j 个块，得到：
- `m_j`：前 j 个块中的行最大值
- `l_j`：前 j 个块的 `sum(exp(s - m_j))` 
- `o_j`：前 j 个块的加权输出 `sum(exp(s - m_j) * v) / l_j`

现在处理第 j+1 个块（新的 S_block），更新公式为：

```
# 1. 更新最大值
m_new = max(m_j, rowmax(S_block))

# 2. 修正旧的累积和
#    旧的 l_j 是在 m_j 下计算的，需要调整到 m_new
l_new = exp(m_j - m_new) * l_j + rowsum(exp(S_block - m_new))

# 3. 修正旧的输出累积
#    旧的 o_j 需要乘以修正因子 (l_j * exp(m_j - m_new)) / l_new
#    新块的贡献需要加上 exp(S_block - m_new) @ V_block / l_new
o_new = (l_j * exp(m_j - m_new) * o_j + exp(S_block - m_new) @ V_block) / l_new
```

### 1.3 在线 Softmax 的关键推导

```
设 x = [x_1, x_2, ..., x_n] 分为两部分: x_A 和 x_B

标准 softmax(x)_i = exp(x_i - m) / sum(exp(x - m))
其中 m = max(x)

假设已处理 x_A，得到:
  m_A = max(x_A)
  l_A = sum(exp(x_A - m_A))

现在处理 x_B:
  m_B = max(x_B)
  m_new = max(m_A, m_B)
  
  l_new = sum(exp(x - m_new))
        = sum(exp(x_A - m_new)) + sum(exp(x_B - m_new))
        = exp(m_A - m_new) * sum(exp(x_A - m_A)) + sum(exp(x_B - m_new))
        = exp(m_A - m_new) * l_A + sum(exp(x_B - m_new))

输出修正:
  旧输出 (未归一化): O_A_unnorm = sum_A(exp(x_i - m_A) * v_i) = l_A * o_A
  调整到 m_new: O_A_unnorm_new = exp(m_A - m_new) * l_A * o_A
  新块贡献:     O_B_unnorm = sum_B(exp(x_i - m_new) * v_i)
  
  o_new = (O_A_unnorm_new + O_B_unnorm) / l_new
```

## 2. FlashAttention v1 算法概览

### 2.1 算法伪代码

```
FlashAttention v1 Forward:

输入: Q [N, d], K [N, d], V [N, d]
输出: O [N, d], L [N] (logsumexp，用于反向传播)

将 Q 分为 T_q = ceil(N / B_q) 个块，每块大小 B_q
将 K, V 分为 T_kv = ceil(N / B_kv) 个块，每块大小 B_kv

对每个 Q 块 i = 0, ..., T_q - 1:
    加载 Q_i [B_q, d] 到 SRAM
    初始化: m_i = [-inf, ...],  l_i = [0, ...],  O_i = [0, ...]
    
    对每个 KV 块 j = 0, ..., T_kv - 1:
        加载 K_j [B_kv, d], V_j [B_kv, d] 到 SRAM
        
        # 计算局部注意力分数
        S_ij = Q_i @ K_j^T  [B_q, B_kv]
        S_ij = S_ij / sqrt(d)
        
        # 在线 softmax 更新
        m_ij = rowmax(S_ij)             [B_q]
        m_new = max(m_i, m_ij)          [B_q]
        
        # 修正因子
        alpha = exp(m_i - m_new)        [B_q]
        P_ij = exp(S_ij - m_new)        [B_q, B_kv]
        
        # 更新 l 和 O
        l_new = alpha * l_i + rowsum(P_ij)  [B_q]
        O_i = alpha[:, None] * O_i + P_ij @ V_j    (未归一化)
        
        m_i = m_new
        l_i = l_new
    
    # 最终归一化
    O_i = O_i / l_i[:, None]
    L_i = m_i + log(l_i)   # logsumexp
    
    写回 O_i, L_i
```

### 2.2 算法图解

```
                    K^T [d, N]                    V [N, d]
               ┌──┬──┬──┬──┬──┐             ┌──────────┐
               │K0│K1│K2│K3│K4│             │ V0       │
               │  │  │  │  │  │             │ V1       │
               │  │  │  │  │  │             │ V2       │
               └──┴──┴──┴──┴──┘             │ V3       │
                                            │ V4       │
   Q [N, d]         S [Bq, Bkv]             └──────────┘
  ┌──────┐                                 
  │ Q0   │──>  Q0@K0^T -> 在线softmax -> + P00@V0
  │      │──>  Q0@K1^T -> 在线softmax -> + P01@V1
  │      │──>  Q0@K2^T -> 在线softmax -> + P02@V2 -> O0 (归一化)
  │      │──>  Q0@K3^T -> 在线softmax -> + P03@V3
  │      │──>  Q0@K4^T -> 在线softmax -> + P04@V4
  ├──────┤
  │ Q1   │──>  同上，处理 Q1 块
  ├──────┤
  │ Q2   │──>  同上，处理 Q2 块
  ├──────┤
  │ ...  │
  └──────┘
  
  关键: 每个 Q 块独立处理 (可并行)
        内层循环遍历所有 KV 块 (串行)
        S 矩阵只有 [Bq, Bkv] 大小 (在 SRAM 中)
```

### 2.3 内存分析

```
标准 Attention:                  FlashAttention v1:
  S 矩阵: O(N^2)               SRAM 中的 S 块: O(B_q * B_kv)
  P 矩阵: O(N^2)               无需存储完整 P
  额外内存: O(N^2)              额外内存: O(N) (仅 m, l)
  HBM 读写: O(N^2)              HBM 读写: O(N^2 * d / M)
                                 其中 M = SRAM 大小
```

在实际 GPU 上，由于 SRAM 通常足够大，FlashAttention 的 HBM 读写
接近 O(N * d)，远小于标准方法的 O(N^2)。

In [ ]:
import torch
import triton
import triton.language as tl
import math

print(f"PyTorch version: {torch.__version__}")
print(f"Triton version: {triton.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. 在线 Softmax 的 Python 验证

在实现 Triton kernel 之前，让我们先用 Python 验证在线 softmax 的正确性。

In [ ]:
def online_softmax_demo(scores_blocks, v_blocks):
    """
    用 Python 演示在线 softmax 在 attention 中的应用。
    
    参数:
        scores_blocks: list of tensors, 每个是 [M, block_n] 的分数块
        v_blocks: list of tensors, 每个是 [block_n, d] 的 V 块
    
    返回:
        output: [M, d] 注意力输出
    """
    M = scores_blocks[0].shape[0]
    d = v_blocks[0].shape[1]
    device = scores_blocks[0].device
    
    # 初始化
    m_i = torch.full((M,), float('-inf'), device=device)  # 行最大值
    l_i = torch.zeros(M, device=device)                    # 分母累积
    o_i = torch.zeros(M, d, device=device)                 # 输出累积 (未归一化)
    
    for s_block, v_block in zip(scores_blocks, v_blocks):
        # 当前块的行最大值
        m_block = s_block.max(dim=-1).values  # [M]
        
        # 新的全局最大值
        m_new = torch.maximum(m_i, m_block)   # [M]
        
        # 修正因子
        alpha = torch.exp(m_i - m_new)        # [M]
        
        # 当前块的 exp(s - m_new)
        p_block = torch.exp(s_block - m_new.unsqueeze(-1))  # [M, block_n]
        
        # 更新分母
        l_new = alpha * l_i + p_block.sum(dim=-1)  # [M]
        
        # 更新输出 (未归一化)
        o_i = alpha.unsqueeze(-1) * o_i + p_block @ v_block
        
        # 更新状态
        m_i = m_new
        l_i = l_new
    
    # 最终归一化
    o_i = o_i / l_i.unsqueeze(-1)
    return o_i


# 验证在线 softmax
torch.manual_seed(42)
M, N, d = 4, 16, 8
BLOCK_N = 4

# 生成完整的 scores 和 V
scores_full = torch.randn(M, N, device='cuda')
v_full = torch.randn(N, d, device='cuda')

# 标准方法
p_standard = torch.softmax(scores_full, dim=-1)
o_standard = p_standard @ v_full

# 在线方法：将 scores 和 V 分块
scores_blocks = [scores_full[:, i:i+BLOCK_N] for i in range(0, N, BLOCK_N)]
v_blocks = [v_full[i:i+BLOCK_N, :] for i in range(0, N, BLOCK_N)]

o_online = online_softmax_demo(scores_blocks, v_blocks)

# 比较
max_diff = (o_standard - o_online).abs().max().item()
print(f"标准 softmax 输出:\n{o_standard}")
print(f"\n在线 softmax 输出:\n{o_online}")
print(f"\n最大误差: {max_diff:.10f}")
print(f"结果匹配: {torch.allclose(o_standard, o_online, atol=1e-5)}")

## 4. FlashAttention v1 Triton 实现

### 4.1 Kernel 设计

在 FlashAttention v1 原始论文中，外层循环遍历 K,V 块，内层循环遍历 Q 块。
但在 Triton 实现中，我们通常让每个 program 处理一个 Q 块，内层循环遍历 K,V 块，
这样更容易实现且更符合 Triton 的编程模型。

(注：FlashAttention v2 对此做了进一步优化，外层改为遍历 Q 块。)

```
Kernel 设计:

Grid: (cdiv(N, BLOCK_M), B * H)
  - 每个 program 负责一个 Q 块 (BLOCK_M 行) 和一个 batch-head

每个 program 的工作:

  1. 加载 Q_block [BLOCK_M, d] 到寄存器
  2. 初始化 m = [-inf], l = [0], O = [0]
  3. for j in range(num_kv_blocks):
       a. 加载 K_j [BLOCK_N, d], V_j [BLOCK_N, d]
       b. S = Q_block @ K_j^T         [BLOCK_M, BLOCK_N]
       c. 在线 softmax 更新 m, l
       d. P = exp(S - m_new)           [BLOCK_M, BLOCK_N]
       e. O = alpha * O + P @ V_j      (修正旧输出 + 新贡献)
  4. O = O / l                          (最终归一化)
  5. 写回 O
```

In [ ]:
@triton.jit
def flash_attention_v1_kernel(
    Q_ptr, K_ptr, V_ptr, O_ptr,
    L_ptr,  # logsumexp 输出，用于反向传播
    stride_qb, stride_qh, stride_qn, stride_qd,
    stride_kb, stride_kh, stride_kn, stride_kd,
    stride_vb, stride_vh, stride_vn, stride_vd,
    stride_ob, stride_oh, stride_on, stride_od,
    stride_lb, stride_lh, stride_ln,
    N: tl.constexpr,
    D: tl.constexpr,
    sm_scale,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_D: tl.constexpr,
    IS_CAUSAL: tl.constexpr,
):
    """
    FlashAttention v1 前向传播 Triton kernel。
    
    每个 program 处理一个 Q 块 (BLOCK_M 行)，
    内层循环遍历所有 K,V 块，使用在线 softmax 累积输出。
    
    关键特点：
    - 不物化 N x N 的注意力矩阵
    - 只在 SRAM 中保存 BLOCK_M x BLOCK_N 的小块
    - 使用在线 softmax 逐步更新
    - 内存复杂度 O(N) 而非 O(N^2)
    """
    # 获取 program 索引
    pid_m = tl.program_id(0)    # Q 块索引 (哪一行块)
    pid_bh = tl.program_id(1)   # batch * head 索引
    
    # 基地址
    q_base = Q_ptr + pid_bh * stride_qh
    k_base = K_ptr + pid_bh * stride_kh
    v_base = V_ptr + pid_bh * stride_vh
    o_base = O_ptr + pid_bh * stride_oh
    l_base = L_ptr + pid_bh * stride_lh
    
    # Q 块的行偏移
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)  # [BLOCK_M]
    offs_d = tl.arange(0, BLOCK_D)                      # [BLOCK_D]
    
    # 加载 Q 块: [BLOCK_M, BLOCK_D]
    q_ptrs = q_base + offs_m[:, None] * stride_qn + offs_d[None, :] * stride_qd
    q_mask = (offs_m[:, None] < N) & (offs_d[None, :] < D)
    q_block = tl.load(q_ptrs, mask=q_mask, other=0.0)
    
    # ======================================
    # 初始化在线 softmax 状态
    # ======================================
    m_i = tl.full([BLOCK_M], value=float('-inf'), dtype=tl.float32)   # 行最大值
    l_i = tl.zeros([BLOCK_M], dtype=tl.float32)                       # 分母累积
    acc = tl.zeros([BLOCK_M, BLOCK_D], dtype=tl.float32)              # 输出累积 (未归一化)
    
    # ======================================
    # 内层循环：遍历所有 K,V 块
    # ======================================
    # 因果注意力：只需要遍历到 j <= i 的位置
    if IS_CAUSAL:
        num_kv_blocks = tl.cdiv((pid_m + 1) * BLOCK_M, BLOCK_N)
    else:
        num_kv_blocks = tl.cdiv(N, BLOCK_N)
    
    for block_n_idx in range(0, num_kv_blocks):
        offs_n = block_n_idx * BLOCK_N + tl.arange(0, BLOCK_N)  # [BLOCK_N]
        
        # 加载 K 块: [BLOCK_N, BLOCK_D]
        k_ptrs = k_base + offs_n[:, None] * stride_kn + offs_d[None, :] * stride_kd
        k_mask = (offs_n[:, None] < N) & (offs_d[None, :] < D)
        k_block = tl.load(k_ptrs, mask=k_mask, other=0.0)
        
        # 计算 S_block = Q_block @ K_block^T  [BLOCK_M, BLOCK_N]
        s_block = tl.dot(q_block, tl.trans(k_block))
        s_block = s_block * sm_scale
        
        # 应用掩码
        # 有效元素掩码
        valid_mask = offs_n[None, :] < N
        s_block = tl.where(valid_mask, s_block, float('-inf'))
        
        # 因果掩码: 只允许 q_pos >= k_pos
        if IS_CAUSAL:
            causal_mask = offs_m[:, None] >= offs_n[None, :]
            s_block = tl.where(causal_mask, s_block, float('-inf'))
        
        # ======================================
        # 在线 softmax 更新
        # ======================================
        
        # 1. 计算当前块的行最大值
        m_block = tl.max(s_block, axis=1)  # [BLOCK_M]
        
        # 2. 更新全局最大值
        m_new = tl.maximum(m_i, m_block)   # [BLOCK_M]
        
        # 3. 计算修正因子
        alpha = tl.exp(m_i - m_new)        # [BLOCK_M]
        
        # 4. 计算当前块的 softmax 分子
        p_block = tl.exp(s_block - m_new[:, None])  # [BLOCK_M, BLOCK_N]
        
        # 5. 更新分母
        l_new = alpha * l_i + tl.sum(p_block, axis=1)  # [BLOCK_M]
        
        # 6. 修正旧的输出累积并加上新贡献
        # acc 存储的是 l_i * o_i (即未归一化的加权和)
        # 修正: acc_new = alpha * acc_old + P_block @ V_block
        acc = alpha[:, None] * acc
        
        # 加载 V 块并累积
        v_ptrs = v_base + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vd
        v_mask = (offs_n[:, None] < N) & (offs_d[None, :] < D)
        v_block = tl.load(v_ptrs, mask=v_mask, other=0.0)
        
        p_block_fp16 = p_block.to(tl.float16)
        acc += tl.dot(p_block_fp16, v_block).to(tl.float32)
        
        # 7. 更新状态
        m_i = m_new
        l_i = l_new
    
    # ======================================
    # 最终归一化
    # ======================================
    acc = acc / l_i[:, None]
    
    # 计算 logsumexp (用于反向传播)
    lse = m_i + tl.log(l_i)  # [BLOCK_M]
    
    # 写回输出
    o_ptrs = o_base + offs_m[:, None] * stride_on + offs_d[None, :] * stride_od
    o_mask = (offs_m[:, None] < N) & (offs_d[None, :] < D)
    tl.store(o_ptrs, acc.to(tl.float16), mask=o_mask)
    
    # 写回 logsumexp
    l_ptrs = l_base + offs_m * stride_ln
    l_mask = offs_m < N
    tl.store(l_ptrs, lse, mask=l_mask)

In [ ]:
def flash_attention_v1(
    q: torch.Tensor,  # [B, H, N, d]
    k: torch.Tensor,  # [B, H, N, d]
    v: torch.Tensor,  # [B, H, N, d]
    causal: bool = False,
) -> tuple:
    """
    FlashAttention v1 前向传播。
    
    返回:
        output: [B, H, N, d]
        lse: [B, H, N] logsumexp (用于反向传播)
    """
    B, H, N, d = q.shape
    assert k.shape == (B, H, N, d)
    assert v.shape == (B, H, N, d)
    
    # 分配输出
    o = torch.empty_like(q)
    lse = torch.empty(B, H, N, device=q.device, dtype=torch.float32)
    
    sm_scale = 1.0 / math.sqrt(d)
    
    # 块大小
    BLOCK_M = 64
    BLOCK_N = 64
    BLOCK_D = triton.next_power_of_2(d)
    
    # 网格
    grid = (triton.cdiv(N, BLOCK_M), B * H)
    
    flash_attention_v1_kernel[grid](
        q, k, v, o, lse,
        q.stride(0), q.stride(1), q.stride(2), q.stride(3),
        k.stride(0), k.stride(1), k.stride(2), k.stride(3),
        v.stride(0), v.stride(1), v.stride(2), v.stride(3),
        o.stride(0), o.stride(1), o.stride(2), o.stride(3),
        lse.stride(0), lse.stride(1), lse.stride(2),
        N=N,
        D=d,
        sm_scale=sm_scale,
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        BLOCK_D=BLOCK_D,
        IS_CAUSAL=causal,
    )
    
    return o, lse


print("FlashAttention v1 kernel 定义完成!")

## 5. 正确性验证

将 FlashAttention v1 的输出与 PyTorch 标准实现进行对比。

In [ ]:
def attention_pytorch_reference(q, k, v, causal=False):
    """PyTorch 参考实现。"""
    B, H, N, d = q.shape
    sm_scale = 1.0 / math.sqrt(d)
    s = torch.matmul(q.float(), k.float().transpose(-2, -1)) * sm_scale
    if causal:
        mask = torch.tril(torch.ones(N, N, device=q.device, dtype=torch.bool))
        s = s.masked_fill(~mask, float('-inf'))
    p = torch.softmax(s, dim=-1)
    o = torch.matmul(p, v.float())
    return o.half()


def test_flash_attention_v1(B, H, N, d, causal=False):
    """测试 FlashAttention v1 正确性。"""
    torch.manual_seed(42)
    q = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)
    k = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)
    v = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)
    
    # 参考实现
    out_ref = attention_pytorch_reference(q, k, v, causal=causal)
    
    # FlashAttention v1
    out_flash, lse = flash_attention_v1(q, k, v, causal=causal)
    
    # 比较
    max_diff = (out_ref - out_flash).abs().max().item()
    mean_diff = (out_ref - out_flash).abs().mean().item()
    
    rtol, atol = 1e-2, 1e-2
    is_close = torch.allclose(out_ref, out_flash, rtol=rtol, atol=atol)
    
    causal_str = "因果" if causal else "全量"
    status = "PASS" if is_close else "FAIL"
    print(f"[{status}] B={B}, H={H}, N={N}, d={d}, {causal_str} | "
          f"最大误差={max_diff:.6f}, 平均误差={mean_diff:.6f}")
    
    return is_close


print("=" * 80)
print("FlashAttention v1 正确性测试")
print("=" * 80)

all_pass = True
for N in [64, 128, 256, 512, 1024]:
    for causal in [False, True]:
        result = test_flash_attention_v1(B=2, H=4, N=N, d=64, causal=causal)
        all_pass = all_pass and result

print("\n" + "=" * 80)
if all_pass:
    print("所有 FlashAttention v1 测试通过!")
else:
    print("存在测试失败!")
print("=" * 80)

## 6. 内存对比：标准 vs FlashAttention

FlashAttention 的关键优势是内存效率。让我们实际对比两种方法的内存消耗。

In [ ]:
def compare_memory(seq_lengths, B=1, H=8, d=64):
    """对比标准 Attention 和 FlashAttention 的内存消耗。"""
    print(f"配置: B={B}, H={H}, d={d}")
    print(f"{'N':>8} | {'标准方法 S矩阵(MB)':>20} | {'FA v1 额外内存(MB)':>20} | {'节省比例':>10}")
    print("-" * 70)
    
    for N in seq_lengths:
        # 标准方法: 需要存储 S [B, H, N, N] 和 P [B, H, N, N]
        standard_mem = 2 * B * H * N * N * 2  # S + P, FP16
        standard_mb = standard_mem / 1e6
        
        # FlashAttention: 只需要 O(N) 额外内存
        # m [B, H, N], l [B, H, N], lse [B, H, N]
        flash_mem = 3 * B * H * N * 4  # FP32
        flash_mb = flash_mem / 1e6
        
        ratio = standard_mb / max(flash_mb, 1e-6)
        print(f"{N:>8} | {standard_mb:>20.2f} | {flash_mb:>20.2f} | {ratio:>9.1f}x")


compare_memory([256, 512, 1024, 2048, 4096, 8192, 16384])

## 7. 性能 Benchmark

对比朴素 Attention 和 FlashAttention v1 的执行时间。

In [ ]:
# 先定义一个简单的 naive attention 用于对比
def naive_attention_pytorch(q, k, v, causal=False):
    """PyTorch 标准 Attention（物化完整 S 矩阵）。"""
    B, H, N, d = q.shape
    sm_scale = 1.0 / math.sqrt(d)
    s = torch.matmul(q, k.transpose(-2, -1)) * sm_scale
    if causal:
        mask = torch.tril(torch.ones(N, N, device=q.device, dtype=torch.bool))
        s = s.masked_fill(~mask, float('-inf'))
    p = torch.softmax(s, dim=-1)
    o = torch.matmul(p, v)
    return o


@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=['N'],
        x_vals=[128, 256, 512, 1024, 2048, 4096],
        line_arg='provider',
        line_vals=['pytorch_naive', 'flash_v1', 'flash_v1_causal'],
        line_names=['PyTorch Naive', 'Flash Attn v1', 'Flash Attn v1 (Causal)'],
        styles=[('blue', '-'), ('red', '-'), ('green', '--')],
        ylabel='ms',
        plot_name='Flash Attention v1 vs Naive Attention',
        args={'B': 2, 'H': 8, 'd': 64},
    )
)
def bench_flash_v1(B, H, N, d, provider):
    q = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)
    k = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)
    v = torch.randn(B, H, N, d, device='cuda', dtype=torch.float16)
    
    quantiles = [0.5, 0.2, 0.8]
    
    if provider == 'pytorch_naive':
        ms, min_ms, max_ms = triton.testing.do_bench(
            lambda: naive_attention_pytorch(q, k, v),
            quantiles=quantiles
        )
    elif provider == 'flash_v1':
        ms, min_ms, max_ms = triton.testing.do_bench(
            lambda: flash_attention_v1(q, k, v, causal=False),
            quantiles=quantiles
        )
    elif provider == 'flash_v1_causal':
        ms, min_ms, max_ms = triton.testing.do_bench(
            lambda: flash_attention_v1(q, k, v, causal=True),
            quantiles=quantiles
        )
    
    return ms, min_ms, max_ms


bench_flash_v1.run(print_data=True)

## 8. FlashAttention v1 算法要点总结

### 8.1 核心创新

```
┌─────────────────────────────────────────────────────────────┐
│                  FlashAttention v1 核心创新                  │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. 分块计算 (Tiling)                                       │
│     - 将 Q, K, V 分成小块                                    │
│     - S 矩阵只在 SRAM 中以 [Bq, Bkv] 大小存在              │
│     - 永远不写回 HBM                                        │
│                                                             │
│  2. 在线 Softmax                                            │
│     - 维护 running max (m) 和 running sum (l)               │
│     - 每处理一个 K 块，更新 m 和 l                           │
│     - 通过修正因子 exp(m_old - m_new) 调整旧结果             │
│                                                             │
│  3. 重计算 (Recomputation)                                   │
│     - 反向传播时不存储 S 和 P 矩阵                          │
│     - 利用保存的 logsumexp 重新计算所需值                    │
│     - 用计算换内存                                          │
│                                                             │
├─────────────────────────────────────────────────────────────┤
│  结果:                                                      │
│  - 内存: O(N^2) -> O(N)  (不考虑输入输出)                    │
│  - IO:   O(N^2) -> O(N^2 * d / M)                          │
│  - 实际速度提升: 2-4x (取决于序列长度)                      │
└─────────────────────────────────────────────────────────────┘
```

### 8.2 v1 的局限性

FlashAttention v1 虽然大幅减少了内存使用，但仍有改进空间：

1. **并行性不足**：v1 的外层循环遍历 K,V 块（串行），Q 块的并行度受限
2. **非矩阵乘法 FLOPS 较多**：大量的 rescaling 操作
3. **Warp 间的工作分配不够优化**

这些问题将在 FlashAttention v2 中得到解决。

## 9. 练习

### 练习 1: 在线 Softmax 手动推导
给定 scores = [1, 2, 3, 4, 5, 6]，分成两块 [1, 2, 3] 和 [4, 5, 6]。
手动计算在线 softmax 的每一步（m, l 的更新），验证最终结果与标准 softmax 一致。

### 练习 2: 不同块大小的影响
修改 BLOCK_M 和 BLOCK_N 的值（如 32, 64, 128），观察对性能的影响。
思考：为什么块太小或太大都不好？

### 练习 3: 输出归一化的时机
在当前实现中，我们在循环结束后才对输出做归一化（除以 l）。
也可以在每一步都归一化（即 o_i 始终是归一化的）。
实现这种 "always-normalized" 版本，并比较数值精度。

### 练习 4: Logsumexp 验证
验证 kernel 输出的 logsumexp 值 L = m + log(l) 是否正确。
提示：L 应该等于 log(sum(exp(s))) 对每一行的结果。

### 练习 5: 支持不同的 Q 和 KV 长度
修改 kernel 支持 Q 的序列长度 N_q 与 K,V 的序列长度 N_kv 不同的情况
（这在 cross-attention 中很常见）。

## 总结

本章我们学习了 FlashAttention v1 的核心思想和实现：

1. **在线 Softmax**：通过维护 running max 和 running sum，实现逐块更新
2. **分块算法**：每个 program 处理一个 Q 块，内层循环遍历 K,V 块
3. **内存效率**：从 O(N^2) 降低到 O(N)，使长序列成为可能
4. **正确性保证**：在线 softmax 数学上等价于标准 softmax

### 核心公式回顾

```
m_new = max(m_old, rowmax(S_block))
alpha = exp(m_old - m_new)
l_new = alpha * l_old + rowsum(exp(S_block - m_new))
O_new = alpha * O_old + exp(S_block - m_new) @ V_block
最终: O = O / l
```

下一章：FlashAttention v2 —— 更好的并行性和更少的非矩阵乘法操作